# Proyecto PC3 — PLN Movistar Perú
## Persona A · Continuación con 146 reclamos (Google Colab)

**Objetivo:** cargar el CSV que ya tienes, revisar su calidad y etiquetas, aplicar limpieza, tokenización, stopwords y lematización, visualizar los datos y generar entregables para la Persona B. **No necesitas volver a subir el Excel de OSIPTEL en este notebook**: aquí trabajamos solamente con los textos.

**Transparencia:** 132 registros son **sintéticos (inventados para practicar)** y 14 son **candidatos de publicaciones públicas pendientes de verificación**. Esto **no sustituye** la meta del curso de 500 reclamos reales etiquetados y revisados. El resultado se llamará `reclamos_practica.csv` para evitar confusión.

### 1. Instalar herramientas de PLN en español

In [ ]:
# Ejecutar esta celda una vez en Google Colab. spaCy + modelo en español.
import sys, subprocess, importlib.util

def instalar(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', *args])

if importlib.util.find_spec('spacy') is None:
    instalar('install', '-q', 'spacy>=3.7,<4')
import spacy
try:
    nlp = spacy.load('es_core_news_sm', disable=['parser','ner'])
except OSError:
    instalar('install', '-q', 'https://github.com/explosion/spacy-models/releases/download/es_core_news_sm-3.8.0/es_core_news_sm-3.8.0-py3-none-any.whl')
    nlp = spacy.load('es_core_news_sm', disable=['parser','ner'])
print('spaCy listo:', spacy.__version__)

### 2. Subir el archivo CORRECTO (.csv)

In [ ]:
from google.colab import files
import io, pandas as pd, numpy as np, re, unicodedata
import matplotlib.pyplot as plt
from collections import Counter

print('Selecciona: reclamos_mixtos_146_para_practica.csv (no el Excel)')
subidos = files.upload()
csvs = [nombre for nombre in subidos if nombre.lower().endswith('.csv')]
if not csvs:
    raise ValueError('No subiste un archivo CSV. Selecciona reclamos_mixtos_146_para_practica.csv')
archivo = 'reclamos_mixtos_146_para_practica.csv' if 'reclamos_mixtos_146_para_practica.csv' in csvs else csvs[0]
df = pd.read_csv(io.BytesIO(subidos[archivo]), encoding='utf-8-sig')
print(f'Cargado: {archivo} | filas: {len(df)}')
display(df.head(5))

### 3. Auditoría de estructura, procedencia y etiquetas
Las categorías se mantienen **exactamente** como están establecidas en el documento del proyecto. Una etiqueta sugerida **no equivale a una validación humana**.

In [ ]:
CATEGORIAS = [
 'Calidad e idoneidad en la prestación del servicio',
 'Facturación y cobro',
 'Instalación, activación o traslado del servicio',
 'Falta de ejecución de baja o suspensión del servicio',
 'Contratación no solicitada',
 'Incumplimiento de condiciones contractuales, ofertas y promociones',
 'Migración', 'Portabilidad', 'Falta de servicio', 'Recargas',
 'Otras materias reclamables'
]
REQUERIDAS = ['id','texto','categoria','fuente','fecha','tipo_dato','url_fuente','estado_etiqueta']
faltantes = [col for col in REQUERIDAS if col not in df.columns]
if faltantes:
    raise ValueError(f'Faltan estas columnas: {faltantes}')

print('Total de registros:',len(df))
print('Origen declarado:')
display(df['tipo_dato'].fillna('(sin tipo)').value_counts().rename_axis('tipo_dato').reset_index(name='cantidad'))
print('Etiquetas según fuente:')
display(pd.crosstab(df['tipo_dato'],df['estado_etiqueta'],dropna=False))
print('Nulos por columna:')
display(df.isna().sum().to_frame('nulos'))
print('Categorías fuera de las 11 acordadas:')
display(df.loc[~df['categoria'].isin(CATEGORIAS),['id','categoria']].head(30))
print('IDs repetidos:',df['id'].duplicated().sum())

### 4. Depuración y control de calidad

In [ ]:
trabajo = df.copy()
trabajo['texto'] = trabajo['texto'].fillna('').astype(str).str.strip()
trabajo['categoria'] = trabajo['categoria'].fillna('').astype(str).str.strip()
# Fusiones permitidas por el documento (sin modificar otras categorías arbitrariamente).
trabajo['categoria'] = trabajo['categoria'].replace({'Otros':'Otras materias reclamables'})

# Casos multi-etiqueta quedan aparte; no se usan en el clasificador de etiqueta única.
multietiqueta = trabajo[trabajo['categoria'].str.contains('|',regex=False)].copy()
trabajo = trabajo[~trabajo['categoria'].str.contains('|',regex=False)].copy()
# Registros inválidos separados, sin ocultarlos.
invalidos = trabajo[(trabajo['texto']=='') | (~trabajo['categoria'].isin(CATEGORIAS))].copy()
trabajo = trabajo[(trabajo['texto']!='') & (trabajo['categoria'].isin(CATEGORIAS))].copy()
# Evita duplicados exactos o con diferencias de mayúsculas/espacios.
trabajo['_clave'] = trabajo['texto'].str.lower().str.replace(r'\s+',' ',regex=True).str.strip()
duplicados = trabajo[trabajo.duplicated('_clave',keep='first')].copy()
trabajo = trabajo.drop_duplicates('_clave',keep='first').drop(columns='_clave')
trabajo = trabajo.reset_index(drop=True)
print('Registros utilizables para práctica:',len(trabajo))
print('Multi-etiqueta excluidos:',len(multietiqueta),'| Inválidos:',len(invalidos),'| Duplicados:',len(duplicados))

### 5. Exploración: categorías, fuentes y longitud de los textos

In [ ]:
trabajo['cantidad_palabras_original'] = trabajo['texto'].str.split().str.len()
fig, ax = plt.subplots(figsize=(11,5))
trabajo['categoria'].value_counts().sort_values().plot.barh(ax=ax)
ax.set(title='Cantidad de reclamos por categoría (base de práctica)',xlabel='Número de textos',ylabel='Categoría')
plt.tight_layout();plt.show()

fig, ax = plt.subplots(figsize=(8,4))
trabajo['tipo_dato'].value_counts().plot.bar(ax=ax)
ax.set(title='Procedencia declarada: sintéticos vs. pendientes de verificación',xlabel='Tipo de dato',ylabel='Registros')
plt.xticks(rotation=15);plt.tight_layout();plt.show()

display(trabajo['cantidad_palabras_original'].describe().to_frame('palabras_por_reclamo'))

### 6. Preprocesamiento textual obligatorio
Transformamos cada texto: **eliminar URL, correo, menciones, números y símbolos/emojis → minúsculas → tokenizar → eliminar stopwords → lematizar** con spaCy. Se conservan negaciones como `no`, `nunca`, `sin`, `ni`, porque son relevantes en reclamos. No se inventan palabras ni se cambian las etiquetas.

In [ ]:
# Aseguramos que el modelo tiene lematizador.
print('Pipeline de spaCy:',nlp.pipe_names)
if 'lemmatizer' not in nlp.pipe_names:
    raise RuntimeError('El modelo español no cargó el lematizador.')

NEGACIONES = {'no','nunca','sin','ni','tampoco','jamás'}
STOPWORDS = set(nlp.Defaults.stop_words) - NEGACIONES

def limpiar_basico(texto):
    texto = unicodedata.normalize('NFKC',str(texto)).lower()
    texto = re.sub(r'https?://\S+|www\.\S+', ' ', texto)
    texto = re.sub(r'[\w.+-]+@[\w.-]+\.[a-z]{2,}', ' ', texto)
    texto = re.sub(r'(?<!\w)@[\w_]+', ' ', texto)
    texto = re.sub(r'\d+', ' ', texto)
    # Conservar letras acentuadas y ñ; retirar puntuación, emojis y símbolos.
    texto = re.sub(r'[^a-záéíóúüñ\s]', ' ', texto)
    return re.sub(r'\s+',' ',texto).strip()

trabajo['texto_normalizado'] = trabajo['texto'].apply(limpiar_basico)
textos = trabajo['texto_normalizado'].tolist()
limpios, tokens_resultado = [], []
for doc in nlp.pipe(textos,batch_size=64):
    tokens = []
    for tok in doc:
        lema = tok.lemma_.lower().strip()
        if not lema or not lema.isalpha() or len(lema) < 2:
            continue
        if lema in STOPWORDS:
            continue
        tokens.append(lema)
    tokens_resultado.append(tokens)
    limpios.append(' '.join(tokens))
trabajo['tokens'] = tokens_resultado
trabajo['texto_limpio'] = limpios
trabajo['cantidad_palabras_limpias'] = trabajo['tokens'].map(len)

print('Ejemplos ANTES y DESPUÉS:')
display(trabajo[['texto','texto_limpio','categoria']].head(10))
print('Textos vacíos después de preprocesamiento:',(trabajo['texto_limpio']=='').sum())

### 7. Análisis exploratorio del texto procesado

In [ ]:
frecuencias = Counter(token for fila in trabajo['tokens'] for token in fila)
print('20 lemas más frecuentes:')
display(pd.DataFrame(frecuencias.most_common(20),columns=['lema','frecuencia']))
fig, ax = plt.subplots(figsize=(10,5))
if frecuencias:
    mas_comunes = pd.DataFrame(frecuencias.most_common(20),columns=['lema','frecuencia'])
    ax.barh(mas_comunes['lema'][::-1],mas_comunes['frecuencia'][::-1])
ax.set(title='20 lemas más frecuentes (base de práctica)',xlabel='Frecuencia')
plt.tight_layout();plt.show()

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(trabajo['cantidad_palabras_limpias'],bins=20)
ax.set(title='Distribución de longitud después de la limpieza',xlabel='Cantidad de tokens',ylabel='Número de reclamos')
plt.tight_layout();plt.show()

### 8. Revisión manual de las etiquetas (obligatoria para la entrega real)
Exportamos una hoja de revisión que puedes abrir en Excel/Google Sheets. **No marcamos las etiquetas como verificadas automáticamente**. Si quieres corregir las categorías, hazlo en este CSV, súbelo en una ejecución posterior y vuelve a correr las etapas 4–10.

In [ ]:
revision = trabajo[['id','texto','categoria','fuente','fecha','tipo_dato','url_fuente','estado_etiqueta']].copy()
revision['categoria_validada_por_humano'] = ''
revision['observaciones'] = ''
revision.to_csv('revision_manual_etiquetas.csv',index=False,encoding='utf-8-sig')
print('Generado revision_manual_etiquetas.csv')
print('Las categorías actuales son propuestas y requieren revisión manual.')

### 9. Exportar los entregables para la Persona B
El esquema mínimo acordado es `id, texto, texto_limpio, categoria, sentimiento, fuente, fecha`. `sentimiento` queda vacío porque lo completa la Persona B. Mantenemos metadatos de procedencia adicionales **en archivo separado**. La base se exporta como **práctica**, ya que mezcla textos sintéticos y 14 candidatos no verificados.

In [ ]:
salida = trabajo[trabajo['texto_limpio'].ne('')].copy()
# Sentimiento corresponde a la Persona B; nunca inventarlo aquí.
salida['sentimiento'] = pd.NA
columnas_oficiales = ['id','texto','texto_limpio','categoria','sentimiento','fuente','fecha']
salida[columnas_oficiales].to_csv('reclamos_practica.csv',index=False,encoding='utf-8-sig')
salida[['id','tipo_dato','url_fuente','estado_etiqueta']].to_csv('procedencia_practica.csv',index=False,encoding='utf-8-sig')
if len(multietiqueta): multietiqueta.to_csv('excluidos_multietiqueta.csv',index=False,encoding='utf-8-sig')
if len(invalidos): invalidos.to_csv('excluidos_invalidos.csv',index=False,encoding='utf-8-sig')
if len(duplicados): duplicados.to_csv('excluidos_duplicados.csv',index=False,encoding='utf-8-sig')
print('Archivo para pruebas de Persona B: reclamos_practica.csv')
print('Metadatos: procedencia_practica.csv')
print('Registros exportados:',len(salida))
display(salida[columnas_oficiales].head(5))

### 10. Reporte breve de fuentes y volumen para tu informe

In [ ]:
from pathlib import Path
volumen = trabajo.groupby(['tipo_dato','fuente'],dropna=False).size().reset_index(name='cantidad')
print('Volumen por procedencia y fuente:')
display(volumen)
lines = [
 'PROYECTO PC3 – PERSONA A – REPORTE DE RECOLECCIÓN Y PREPROCESAMIENTO',
 f'Archivo de entrada: {archivo}',
 f'Registros originales: {len(df)}',
 f'Registros válidos tras depuración: {len(trabajo)}',
 f'Registros exportados tras preprocesamiento: {len(salida)}',
 f'Duplicados descartados: {len(duplicados)}',
 f'Inválidos descartados: {len(invalidos)}',
 f'Multi-etiqueta separados: {len(multietiqueta)}',
 '', 'PROCEDENCIA DECLARADA (NO EQUIVALE A VERIFICACIÓN DE FUENTES):'
]
lines += [f'- {r.tipo_dato} / {r.fuente}: {r.cantidad}' for r in volumen.itertuples(index=False)]
lines += [
 '', 'METODOLOGÍA:',
 '- Datos mixtos: 132 textos sintéticos de práctica y 14 candidatos obtenidos de fuentes públicas, pendientes de verificar.',
 '- Se controlaron nulos, clases permitidas, textos multietiqueta y duplicados.',
 '- Limpieza con expresiones regulares, minúsculas, tokenización, stopwords y lematización en español (spaCy).',
 '- Las categorías son etiquetas propuestas; deben verificarse manualmente.',
 '- Sentimiento no se procesa en la Persona A; corresponde a Persona B.',
 '', 'LIMITACIONES:',
 '- La base no cumple la meta de 500 reclamos reales etiquetados del proyecto.',
 '- No se deben reportar evaluaciones de modelos sobre datos sintéticos como resultados con validez en el mundo real.'
]
Path('reporte_persona_A_practica.txt').write_text('\n'.join(lines),encoding='utf-8')
print('\n'.join(lines))

### 11. Descargar archivos generados

In [ ]:
import zipfile, os
archivos = ['reclamos_practica.csv','procedencia_practica.csv','revision_manual_etiquetas.csv','reporte_persona_A_practica.txt']
archivos += [x for x in ['excluidos_multietiqueta.csv','excluidos_invalidos.csv','excluidos_duplicados.csv'] if os.path.exists(x)]
with zipfile.ZipFile('entregables_persona_A_practica.zip','w',zipfile.ZIP_DEFLATED) as z:
    for a in archivos: z.write(a)
files.download('entregables_persona_A_practica.zip')

## ¿Qué enviar al equipo?
Comparte con la Persona B el archivo **`reclamos_practica.csv`** para realizar pruebas de modelos, y acompáñalo de **`procedencia_practica.csv`**. Para la entrega final del curso faltará recolectar y verificar suficientes textos reales, completar la revisión humana de categorías y repetir esta misma limpieza con esos registros. El documento original exige **500 textos etiquetados como mínimo**.